# Inspect the shortest IQPE iteration circuit

This notebook complements the iterative phase-estimation chapter by constructing and rendering the power-one iteration circuit. It does not execute the simulator.

Keep `tutorial_choose_active_space.py`, `tutorial_map_n2_to_qubits.py`, `tutorial_prepare_trial_state.py`, and `tutorial_run_iqpe.py` in the same folder as this notebook.

## Import the shared workflow

Use the tested Chapter 6 function to construct the Hamiltonian, trial state, evolution-time choice, and six IQPE iteration circuits.

The import cell silences QDK/Chemistry library logs so the circuit data remain easy to read. Change `Logger.LogLevel.off` to `Logger.LogLevel.info` and rerun the cell to see detailed calculation logs.

In [ ]:
import json

from IPython.display import Markdown, display
from qdk.widgets import Circuit
from qdk_chemistry.utils import Logger
from tutorial_prepare_trial_state import circuit_statistics
from tutorial_run_iqpe import prepare_iqpe_problem

Logger.set_global_level(Logger.LogLevel.off)

## Construct the iteration circuits



The circuit builder returns powers 32, 16, 8, 4, 2, and 1 in that order. Compare all six circuit dimensions, then select the last circuit because its power-one controlled evolution has the fewest decomposed operations.

In [ ]:
problem = prepare_iqpe_problem()

circuit_powers = [32, 16, 8, 4, 2, 1]

circuit_rows = [
    (power, *circuit_statistics(circuit)[:2])
    for power, circuit in zip(
        circuit_powers, problem.iteration_circuits, strict=True
    )
]

table_rows = [
    "| Controlled power | Logical qubits | Decomposed logical gates |",
    "|---:|---:|---:|",
]

for power, qubits, gates in circuit_rows:
    table_rows.append(f"| {power} | {qubits} | {gates:,} |")

display(Markdown("\n".join(table_rows)))

shortest_circuit = problem.iteration_circuits[-1]

num_qubits, num_gates, gate_counts = circuit_statistics(shortest_circuit)

## Register roles

The circuit viewer numbers wires from top to bottom:

- **q0** is the readout ancilla. Its H gates, feedback rotation, controlled evolution, and measurement extract one phase bit.

- **q1–q12** are the compute register. They hold the encoded molecular trial state and receive the controlled Hamiltonian evolution.

The readout ancilla is algorithm workspace; it is not a thirteenth molecular spin orbital.

## Render the shortest circuit

The complete decomposed circuit is long because it contains one state preparation and one controlled first-order Trotter sequence for a 247-term Hamiltonian. Use the viewer's pan and zoom controls to inspect the ancilla operations and the controlled gates connecting q0 to the compute register.

In [ ]:
display(Circuit(shortest_circuit.get_qsharp_circuit()))

## Validate the circuit structure

These assertions verify the shared workflow settings, register size, and shortest-circuit selection without executing a quantum simulation.

In [ ]:
circuit_data = json.loads(shortest_circuit.get_qsharp_circuit().json())

grid_index = int(problem.evolution_time.grid_bitstring, 2)

assert len(problem.iteration_circuits) == problem.num_phase_bits == 6
assert [row[0] for row in circuit_rows] == [32, 16, 8, 4, 2, 1]
assert all(row[1] == 13 for row in circuit_rows)
assert all(
    larger[2] > smaller[2]
    for larger, smaller in zip(circuit_rows, circuit_rows[1:])
)
assert problem.mapping.qubit_hamiltonian.num_qubits == 12
assert len(circuit_data["qubits"]) == 13
assert problem.evolution_time.grid_phase_fraction == grid_index / 2**problem.num_phase_bits
assert num_gates == min(row[2] for row in circuit_rows)
assert sum(gate_counts.values()) == num_gates

## Interpretation

Identify the trial-state preparation region, the readout-ancilla H and feedback operations, the controlled evolution, and the final ancilla measurement. Explain why the compute-register width stays fixed across all six iteration circuits while their controlled-evolution lengths differ.